# Minimaler eigener Runner fuer Task 44

Dieses Notebook ist der Schritt nach dem Human-Agent-Test. Ziel ist nicht LLM-Intelligenz, sondern der Nachweis, dass ein eigener Runner die zwei wichtigen Dateien erzeugen kann:

- `agent_response.json`
- `network.har`

Danach bewertet WebArena-Verified den Run mit `eval-tasks`.

In [1]:
from pathlib import Path
import json
import subprocess

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
OFFICIAL_REPO = PROJECT_ROOT / 'external' / 'webarena-verified'
OFFICIAL_REPO

PosixPath('/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified')

## 1. Voraussetzung: Demo-GitLab laeuft

Falls noch nicht gestartet:

```bash
cd external/webarena-verified
uv run invoke -r examples gitlab-start
```

Der Runner erwartet `http://localhost:8012`.

In [2]:
subprocess.run(['docker', 'ps', '--filter', 'name=wa-demo-gitlab'], check=True)

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


CompletedProcess(args=['docker', 'ps', '--filter', 'name=wa-demo-gitlab'], returncode=0)

## 2. Task-Input fuer Task 44 erzeugen

Der Runner liest `output/tasks.demo.json`. Diese Datei kann jederzeit neu erzeugt werden.

In [3]:
(OFFICIAL_REPO / 'output').mkdir(exist_ok=True)
subprocess.run([
    'uv', 'run', 'webarena-verified', 'agent-input-get',
    '--task-ids', '44',
    '--config', 'examples/configs/config.demo.json',
    '--output', 'output/tasks.demo.json',
], cwd=OFFICIAL_REPO, check=True)

json.loads((OFFICIAL_REPO / 'output/tasks.demo.json').read_text())

[WebArena Verified] [INFO] Loading config from: '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/examples/configs/config.demo.json'
[WebArena Verified] [INFO] Using test_data_file: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/assets/dataset/webarena-verified.json
[WebArena Verified] [INFO] No config provided, using default configuration
[WebArena Verified] [INFO] Using test_data_file: /Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/assets/dataset/webarena-verified.json
[WebArena Verified] [INFO] Loading tasks from: '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified/assets/dataset/webarena-verified.json'
[WebArena Verified] [INFO] Loaded 812 tasks successfully.
[WebArena Verified] [INFO] WebArenaVerified initialized successfully
[WebArena Verified] [INFO] Wrote 1 agent inputs to output/tasks.demo.json


[{'sites': ['gitlab'],
  'task_id': 44,
  'intent_template_id': 303,
  'start_urls': ['http://localhost:8012'],
  'intent': 'Open my todos page'}]

## 3. Eigenen Runner ausfuehren

Der Runner macht absichtlich nur das Minimum fuer Task 44:

1. Task laden.
2. Browser starten.
3. Von `http://localhost:8012` nach `/dashboard/todos` navigieren.
4. HAR speichern.
5. `agent_response.json` mit `NAVIGATE/SUCCESS` schreiben.
6. `eval-tasks` starten.

Mit `tqdm` siehst du die Schritte.

In [4]:
subprocess.run([
    str(PROJECT_ROOT / '.venv/bin/python'),
    str(PROJECT_ROOT / 'scripts/run_gitlab_task44_navigate_runner.py'),
    '--repo-root', str(OFFICIAL_REPO),
    '--tasks-file', 'output/tasks.demo.json',
    '--task-id', '44',
    '--output-root', 'output/auto-run',
    '--config', 'examples/configs/config.demo.json',
], cwd=PROJECT_ROOT, check=True)

Task 44:  25%|██▌       | 1/4 [00:00<00:00, 75.05step/s]
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/urllib/request.py", line 1344, in do_open
    h.request(req.get_method(), req.selector, req.data, headers,
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1358, in request
    self._send_request(method, url, body, headers, encode_chunked)
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1404, in _send_request
    self.endheaders(body, encode_chunked=encode_chunked)
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1353, in endheaders
    self._send_output(message_body, encode_chunked=encode_chunked)
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/F

CalledProcessError: Command '['/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/.venv/bin/python', '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/scripts/run_gitlab_task44_navigate_runner.py', '--repo-root', '/Users/niclascramer/Privat/Uni/Uni-Reutlingen/Masterarbeit/05_Code/external/webarena-verified', '--tasks-file', 'output/tasks.demo.json', '--task-id', '44', '--output-root', 'output/auto-run', '--config', 'examples/configs/config.demo.json']' returned non-zero exit status 1.

## 4. Artefakte inspizieren

In [5]:
run_dir = OFFICIAL_REPO / 'output/auto-run/44'
sorted(p.name for p in run_dir.iterdir())

['agent_response.json', 'eval_result.json', 'network.har']

In [6]:
json.loads((run_dir / 'agent_response.json').read_text())

{'task_type': 'NAVIGATE',
 'status': 'SUCCESS',
 'retrieved_data': None,
 'error_details': None}

In [7]:
eval_result = json.loads((run_dir / 'eval_result.json').read_text())
{key: eval_result.get(key) for key in ['task_id', 'status', 'score']}

{'task_id': 44, 'status': 'success', 'score': 1.0}

## 5. Was bedeutet das fuer den naechsten Schritt?

Wenn dieser Runner `score = 1.0` erreicht, hast du den Human-Agent fuer eine einfache Aufgabe durch eigenen Code ersetzt. Danach kann derselbe Runner schrittweise erweitert werden:

- mehrere Tasks laden
- Schleife ueber Tasks mit `tqdm`
- BrowserGym/AgentLab statt direktem Playwright
- spaeter Planner, Validator, `H`, `k` und Prozessmetriken